# 01 — Interactive Exploration: Embeddings, ChromaDB & RAG

This notebook lets you interactively experiment with every core layer of the **Document Intelligence RAG System**:
1. **Vector Embeddings** (`all-MiniLM-L6-v2`)
2. **Cosine Similarity Matrix** between sentences
3. **Sliding-Window Document Chunking**
4. **Semantic Retrieval & RAG Prompt Construction**

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so we can import from `src`
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.embeddings import load_embedding_model, generate_embeddings, cosine_similarity
from src.document_loader import chunk_text
from src.vectorstore import VectorStore
from src.rag import RAGPipeline

## 1. Visualizing Embeddings & Cosine Similarity

In [ ]:
model = load_embedding_model()

sentences = [
    "RAG reduces hallucinations by grounding LLMs in external documents.",
    "Retrieval-Augmented Generation retrieves facts from a vector database.",
    "LoRA freezes base weights and trains low-rank decomposition matrices.",
    "I baked a chocolate cake with vanilla frosting yesterday."
]

vectors = generate_embeddings(model, sentences)
print(f"Vector matrix shape: {vectors.shape} (4 sentences x 384 dimensions)\n")

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f"Similarity ({i} vs {j}): {sim:.4f}")
        print(f"  [{i}] {sentences[i]}")
        print(f"  [{j}] {sentences[j]}\n")

## 2. Experimenting with Sliding-Window Chunking

In [ ]:
sample_text = (
    "Large Language Models are trained on massive corpora of text, but their knowledge "
    "is frozen at the time of training. When asked about private company data or recent "
    "events, they may hallucinate plausible-sounding falsehoods. Retrieval-Augmented "
    "Generation solves this by retrieving relevant chunks from a vector database first."
)

chunks = chunk_text(sample_text, chunk_size=120, chunk_overlap=30)
for idx, c in enumerate(chunks):
    print(f"--- Chunk {idx} ({len(c)} chars) ---")
    print(c)
    print()

## 3. Running a Live RAG Query

In [ ]:
pipeline = RAGPipeline()
response = pipeline.answer("How does HNSW speed up vector search?")

print("ANSWER:\n", response.answer)
print("\nSOURCES:", response.sources)